<a href="https://colab.research.google.com/github/JeffersonRodrigues9/Automacao_com_python/blob/main/Extra%C3%A7%C3%A3o_XML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Extração automatizada de informações em documentos XML (NFe e CTe)

import os
import xml.etree.ElementTree as ET
import pandas as pd
from multiprocessing import Pool, cpu_count
from tqdm import tqdm


def formatar_data(data_iso):
    try:
        if len(data_iso) >= 10:
            ano, mes, dia = data_iso[:10].split("-")
            return f"{dia}/{mes}/{ano}"
    except Exception:
        pass
    return ""


def formatar_cnpj_cpf(valor):
    if not valor:
        return ""

    valor = ''.join(filter(str.isdigit, valor))

    if len(valor) == 11:
        return f"{valor[:3]}.{valor[3:6]}.{valor[6:9]}-{valor[9:]}"
    elif len(valor) == 14:
        return f"{valor[:2]}.{valor[2:5]}.{valor[5:8]}/{valor[8:12]}-{valor[12:]}"

    return valor


def get_text_safe(element, tag, ns):
    try:
        el = element.find(tag, ns)
        return el.text if el is not None else ''
    except Exception:
        return ''


def extrair_nfe(root):
    ns = {'nfe': 'http://www.portalfiscal.inf.br/nfe'}

    ide = root.find('.//nfe:ide', ns)
    emit = root.find('.//nfe:emit', ns)
    dest = root.find('.//nfe:dest', ns)
    total = root.find('.//nfe:ICMSTot', ns)

    data_emissao = formatar_data(
        get_text_safe(ide, 'nfe:dhEmi', ns)
    )

    valor_total = get_text_safe(
        total,
        'nfe:vNF',
        ns
    )

    cnpj_empresa = formatar_cnpj_cpf(
        get_text_safe(emit, 'nfe:CNPJ', ns)
    )

    nome_empresa = get_text_safe(
        emit,
        'nfe:xNome',
        ns
    )

    cpf_cnpj_pessoa = get_text_safe(dest, 'nfe:CNPJ', ns)

    if not cpf_cnpj_pessoa:
        cpf_cnpj_pessoa = get_text_safe(dest, 'nfe:CPF', ns)

    cpf_cnpj_pessoa = formatar_cnpj_cpf(
        cpf_cnpj_pessoa
    )

    nome_pessoa = get_text_safe(
        dest,
        'nfe:xNome',
        ns
    )

    return [{
        'Data de Emissão': data_emissao,
        'CPF/CNPJ': cpf_cnpj_pessoa,
        'Nome Completo': nome_pessoa,
        'CNPJ Empresa': cnpj_empresa,
        'Nome Empresa': nome_empresa,
        'Valor Total': valor_total
    }]


def extrair_cte(root):
    ns = {'cte': 'http://www.portalfiscal.inf.br/cte'}

    ide = root.find('.//cte:ide', ns)
    emit = root.find('.//cte:emit', ns)
    dest = root.find('.//cte:dest', ns)
    vPrest = root.find('.//cte:vPrest', ns)

    data_emissao = formatar_data(
        get_text_safe(ide, 'cte:dhEmi', ns)
    )

    valor_total = get_text_safe(
        vPrest,
        'cte:vTPrest',
        ns
    )

    cnpj_empresa = formatar_cnpj_cpf(
        get_text_safe(emit, 'cte:CNPJ', ns)
    )

    nome_empresa = get_text_safe(
        emit,
        'cte:xNome',
        ns
    )

    cpf_cnpj_pessoa = formatar_cnpj_cpf(
        get_text_safe(dest, 'cte:CNPJ', ns)
    )

    nome_pessoa = get_text_safe(
        dest,
        'cte:xNome',
        ns
    )

    return [{
        'Data de Emissão': data_emissao,
        'CPF/CNPJ': cpf_cnpj_pessoa,
        'Nome Completo': nome_pessoa,
        'CNPJ Empresa': cnpj_empresa,
        'Nome Empresa': nome_empresa,
        'Valor Total': valor_total
    }]


def identificar_tipo(root):
    tag = root.tag.lower()

    if "nfe" in tag:
        return "nfe"

    if "cte" in tag:
        return "cte"

    return "desconhecido"


def processar_xml(xml_path):
    if "nfse" in xml_path.lower():
        return None

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        tipo = identificar_tipo(root)

        if tipo == "nfe":
            return extrair_nfe(root)

        elif tipo == "cte":
            return extrair_cte(root)

        return None

    except Exception:
        return None


def processar_pasta(pasta_xml):
    arquivos = [
        os.path.join(pasta_xml, f)
        for f in os.listdir(pasta_xml)
        if f.lower().endswith('.xml')
    ]

    with Pool(processes=cpu_count()) as pool:
        resultados = list(
            tqdm(
                pool.imap(processar_xml, arquivos),
                total=len(arquivos),
                desc="Processando XMLs"
            )
        )

    resultados = list(filter(None, resultados))

    dados_achatados = [
        item
        for sublist in resultados
        for item in sublist
    ]

    return pd.DataFrame(dados_achatados)


if __name__ == "__main__":

    pasta = r""

    df = processar_pasta(pasta)

    df.to_excel(
        r"",
        index=False
    )

    print("Processo concluído com sucesso!")